# Step 3 — Prior team's baseline (Hybrid NCF + LSTM) and gap diagnosis

Re-implements the previous team's model, evaluates it on the same full slates as every other model, and measures the structural reasons it underperforms.

### Setup (run first)

**On Google Colab:** *Runtime → Change runtime type → GPU* (T4 is enough; A100/L4 is faster). Put the two raw JDsearch
files in `MyDrive/bt4222/raw/` (`user_behavior_data.txt`, `product_meta_data.txt`). The cell mounts Drive, clones the
repository, installs the requirements and writes every output to `MyDrive/bt4222/outputs/`, so results persist between
sessions and the notebooks can be run one after another (or on different days).

**Locally:** run the notebook from the repository with the virtual environment active; data is read from `data/raw/`
and outputs go to `outputs/`.

`CONFIG = "configs/smoke.yaml"` runs everything on a 2,000-user sample with tiny models (minutes, CPU) to check the code;
`configs/default.yaml` is the real run. Every stage skips work whose outputs already exist; pass `"--force"` to redo one.

In [1]:
import os, subprocess, sys
from pathlib import Path

CONFIG = "configs/default.yaml"          # "configs/smoke.yaml" for a quick check
REPO_URL = "https://github.com/clarequek/bt4222-two-tower-recsys.git"
DRIVE_DIR = "/content/drive/MyDrive/bt4222"   # Colab: raw/ (input) and outputs/ (written by the pipeline)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/bt4222-two-tower-recsys")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "-q"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

from jds.notebook import Runner, banner
run = Runner(CONFIG, drive_dir=DRIVE_DIR if IN_COLAB else None)

config configs/default.yaml | scope full | raw data /home/ubuntu/bt4222-two-tower-recsys/data/raw | outputs /home/ubuntu/bt4222-two-tower-recsys/outputs/full


## 3.1 The model (`scripts/02c_prior_ncf.py`, `src/jds/prior_ncf.py`)

Their setup: per-candidate 4-class classification trained on lists subsampled to 1 positive : 4 negatives; user = 2-layer
LSTM + attention over the last 200 history events plus 11 user statistics (including click/cart/order ratios over the whole
history); candidate = item id + brand/shop/category embeddings. Two variants:
* `published` — as released: the attribute lookup indexes the product table with the *history* item index (a bug),
  ranked by the max class probability;
* `fixed` — the item's own attributes, ranked by the expected label.

Both are scored on the full displayed slates of test users, and on their own sampled 1:4 lists for reference (which is
why their originally reported numbers look much better than on the real task).

In [2]:
run.stage("02c_prior_ncf.py")

$ /home/ubuntu/bt4222-two-tower-recsys/.venv/bin/python -u scripts/02c_prior_ncf.py --config configs/default.yaml


06:02:26 prior_ncf | prior model gap diagnosis: {'uses_test_query': False, 'displayed_items_without_item_embedding (cold, not in history >= min_item_count)': 0.683585356901684, 'positives_without_item_embedding': 0.38628913932507103, 'published_variant_wrong_attributes_share (items with metadata)': 0.99999934388928, 'published_variant_wrong_attributes_share_positives': 1.0, 'train_items_per_list_sampled': 9.283108413038098, 'test_items_per_slate_full': 89.15837894491673, 'history_truncated_to_last_events': 200, 'users_with_history_longer_than_maxlen': 0.2332898044652565, 'users_with_history': 173831, 'user_meta_features': ['nunique_cats_1', 'nunique_cats_2', 'nunique_cats_3', 'nunique_cats_4', 'history_len', 'click_ratio', 'cart_ratio', 'order_ratio', 'avg_gap', 'std_gap', 'max_gap'], 'note': 'click/cart/order ratios over the whole history mix retention windows (see eda.json recency_confound)'}


06:02:50 prior_ncf | [fixed] epoch 1: train loss 1.3315 val loss 1.2874 (21s)


06:03:10 prior_ncf | [fixed] epoch 2: train loss 1.2804 val loss 1.2673 (20s)


06:03:31 prior_ncf | [fixed] epoch 3: train loss 1.2525 val loss 1.2556 (20s)


...

│ Prior team Hybrid NCF+LSTM   ┆ 0.135  ┆ 0.0597 ┆ 0.122  ┆ … ┆ 0.2812 ┆ 0.2055 ┆ 0.5286 ┆ 17174   │


│ (pu…                         ┆        ┆        ┆        ┆   ┆        ┆        ┆        ┆         │


│ Prior team Hybrid NCF+LSTM   ┆ 0.4524 ┆ 0.2429 ┆ 0.5402 ┆ … ┆ 0.9787 ┆ 0.9013 ┆ 0.9978 ┆ 17174   │


│ (pu…                         ┆        ┆        ┆        ┆   ┆        ┆        ┆        ┆         │


└──────────────────────────────┴────────┴────────┴────────┴───┴────────┴────────┴────────┴─────────┘


In [3]:
banner()
import pandas as pd
pd.DataFrame(run.json("prior_ncf.json")).style.format(precision=4).highlight_max(subset=["mrr", "p1", "ndcg10", "gndcg10", "hr10"], color="#c6efce")

,model,mrr,p1,ndcg10,gndcg10,hr10,r10,r50,n_users
0,Prior team Hybrid NCF+LSTM (fixed) [test],0.1708,0.0734,0.1644,0.1619,0.3706,0.2851,0.7288,17174
1,"Prior team Hybrid NCF+LSTM (fixed), their sampled 1:4 lists [reference]",0.5545,0.3378,0.6265,0.6157,0.9926,0.9231,0.9982,17174
2,Prior team Hybrid NCF+LSTM (published) [test],0.1350,0.0597,0.1220,0.1195,0.2812,0.2055,0.5286,17174
3,"Prior team Hybrid NCF+LSTM (published), their sampled 1:4 lists [reference]",0.4524,0.2429,0.5402,0.5303,0.9787,0.9013,0.9978,17174


## 3.2 Gap diagnosis

The measured structural gaps (`results/prior_ncf_gaps.json`): the test query is never an input; share of displayed items
(and of positives) with no item embedding because they never occur in history; how often the `published` lookup gives an
item someone else's attributes; training lists of ~5 items vs ~89-item test slates; history truncated to 200 events;
user ratios that mix retention windows (see step 1, recency confound).

In [4]:
run.json("prior_ncf_gaps.json")

{'uses_test_query': False,
 'displayed_items_without_item_embedding (cold, not in history >= min_item_count)': 0.683585356901684,
 'positives_without_item_embedding': 0.38628913932507103,
 'published_variant_wrong_attributes_share (items with metadata)': 0.99999934388928,
 'published_variant_wrong_attributes_share_positives': 1.0,
 'train_items_per_list_sampled': 9.283108413038098,
 'test_items_per_slate_full': 89.15837894491673,
 'history_truncated_to_last_events': 200,
 'users_with_history_longer_than_maxlen': 0.2332898044652565,
 'users_with_history': 173831,
 'user_meta_features': ['nunique_cats_1',
  'nunique_cats_2',
  'nunique_cats_3',
  'nunique_cats_4',
  'history_len',
  'click_ratio',
  'cart_ratio',
  'order_ratio',
  'avg_gap',
  'std_gap',
  'max_gap'],
 'note': 'click/cart/order ratios over the whole history mix retention windows (see eda.json recency_confound)'}